**Create Dataset**

In [1]:
TRAINING_DATA_PATH = "training_data/synth_train_data.npz"

In [ ]:
import librosa
import torch
import numpy as np
from helper_functions import AudioRecordingDataset, generate_dataset

# Generate synthetic data, X = Features, Y = labels. 


X, Y, HV = generate_dataset()


/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
np.savez(file = TRAINING_DATA_PATH, X = X,Y = Y, HV = HV)

**Load Torch Dataset/Dataloader**

In [11]:
from torch.utils.data import DataLoader, Subset
import torch.nn as nn
import torch.optim as optim

data = AudioRecordingDataset(TRAINING_DATA_PATH)
val_mask = (HV == 0.25)
val_idx = np.where(val_mask)[0]
train_idx = np.where(~val_mask)[0]

train_data = Subset(data, train_idx)
val_data = Subset(data, val_idx)

print(f"train {len(train_data)} | val {len(val_data)}")

train_loader = DataLoader(
    dataset = train_data,
    batch_size = 32,
    shuffle = True
)
val_loader = DataLoader(
    dataset = val_data,
    batch_size = 32,
    shuffle = True
)

# batch size x feature_size (64, 84)
model = nn.Sequential(
    nn.Linear(84, 125), 
    nn.ReLU(),
    nn.Linear(125, 88) # 88 = num of valid midi_notes
)
# batch size * output_dim (64, 88)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = 0.01)

train 1760 | val 440


In [12]:
num_epochs = 25

for epoch in range(num_epochs):
    model.train() 
    total_loss = 0
    for batch_idx, (batch_X, batch_Y) in enumerate(train_loader):
        logits = model(batch_X)
        loss = criterion(logits, batch_Y)
        #back pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0,0,0
    with torch.no_grad():
        for batch_X, batch_Y in val_loader:
            logits = model(batch_X)
            val_loss += criterion(logits, batch_Y).item()
            total += batch_Y.size(0)
            correct += (logits.argmax(dim=1) == batch_Y).sum().item()

    print(f"Epoch {epoch+1:3d} | train {total_loss/len(train_loader):.3f} "
          f"| val {val_loss/len(val_loader):.3f}| acc {correct/total:.3f}")


Epoch   1 | train 5.033 | val 2.916| acc 0.368
Epoch   2 | train 2.576 | val 2.048| acc 0.559
Epoch   3 | train 1.883 | val 1.847| acc 0.602
Epoch   4 | train 1.469 | val 1.179| acc 0.743
Epoch   5 | train 1.048 | val 0.925| acc 0.816
Epoch   6 | train 0.822 | val 0.550| acc 0.877
Epoch   7 | train 0.596 | val 0.500| acc 0.891
Epoch   8 | train 0.505 | val 0.387| acc 0.925
Epoch   9 | train 0.614 | val 0.536| acc 0.841
Epoch  10 | train 0.405 | val 0.563| acc 0.893
Epoch  11 | train 0.352 | val 0.456| acc 0.895
Epoch  12 | train 0.343 | val 0.274| acc 0.952
Epoch  13 | train 0.228 | val 0.417| acc 0.916
Epoch  14 | train 0.266 | val 0.208| acc 0.964
Epoch  15 | train 0.391 | val 0.277| acc 0.936
Epoch  16 | train 0.308 | val 0.161| acc 0.968
Epoch  17 | train 0.323 | val 0.261| acc 0.934
Epoch  18 | train 0.251 | val 0.179| acc 0.959
Epoch  19 | train 0.200 | val 0.962| acc 0.841
Epoch  20 | train 0.284 | val 0.190| acc 0.968
Epoch  21 | train 0.160 | val 0.210| acc 0.941
Epoch  22 | t

TODO: Add commenting to all refactored functions (params, returns, brief)

In [ ]:
#%----- Bin # to Note checker-----

# Verify that peak bin corresponds to A4 (440 Hz)
fmin = librosa.note_to_hz('C1')  # 32.7 Hz #lowest bin frequency
freqs = librosa.cqt_frequencies(n_bins=84, fmin=fmin, bins_per_octave=12)
peak_bin = 45 # CHANGE THIS VALUE FOR CHECKER
peak_freq = freqs[peak_bin]
a4_freq = librosa.note_to_hz('A4')  # 440 Hz

print(f"Peak bin: {peak_bin}")
print(f"Peak frequency: {peak_freq:.2f} Hz")
print(f"A4 frequency: {a4_freq:.2f} Hz")
print(f"Difference: {abs(peak_freq - a4_freq):.2f} Hz")
print(f"✓ Phase 1 verified: Peak bin corresponds to A4" if abs(peak_freq - a4_freq) < 5 else "✗ Peak bin does NOT match A4")



In [ ]:
#%----General Plotting-----$#
import librosa.display
import matplotlib.pyplot as plt

# Create subplots: CQT (freq vs amp) and waveform (time vs amp)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Plot CQT spectrogram (frequency vs amplitude)
img = librosa.display.specshow(amp_to_db(cqt, ref=np.max(cqt)), sr=sr, x_axis='time', y_axis='cqt_hz', ax=ax1)
ax1.set_title('CQT Spectrogram (Frequency vs Amplitude)')
fig.colorbar(img, ax=ax1, format='%+2.0f dB')

# Plot waveform (time vs amplitude)
librosa.display.waveshow(y, sr=sr, ax=ax2)
ax2.set_title('Waveform (Time vs Amplitude)')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

In [ ]:
#%----CQT Line Graph
# Plot CQT as line graph (magnitude vs frequency)
fig, ax = plt.subplots(figsize=(12, 6))

# Average CQT magnitude across time to get overall frequency content
cqt_mag = np.abs(cqt)
cqt_mean = np.mean(cqt_mag, axis=1)

# Get frequency values for CQT bins
freqs = librosa.cqt_frequencies(n_bins=cqt.shape[0], fmin=librosa.note_to_hz('C1'), bins_per_octave=12)

# Plot as line graph
ax.plot(freqs, librosa.amplitude_to_db(cqt_mean, ref=np.max(cqt_mean)), linewidth=1.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Amplitude (dB)')
ax.set_title('CQT Line Graph (Frequency vs Amplitude)')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()